# ARCHIVE STUDIO AI no Google Colab

1. **Ambiente de execução → Alterar o tipo → GPU T4** (ou melhor).
2. No ícone 🔑 (**Secrets**) do Colab, crie e ative o "Acesso ao notebook":
   - `GITHUB_TOKEN` — **obrigatório se o repositório for privado**. Crie em github.com → Settings → Developer settings → Personal access tokens → *Fine-grained tokens* → acesso só ao repositório ARCHIVE-STUDIO-AI com permissão **Contents: Read-only**.
   - `GEMINI_API_KEY` (opcional, grátis em aistudio.google.com/apikey) e `HF_TOKEN` (opcional).
3. Rode as 3 células em ordem. A última mostra o painel e um link externo.

Na T4 (15 GB), use **SDXL** ou **Z-Image Turbo** (perfil Econômico) para imagens e **LTX-Video** para vídeo por IA.
A ponte com o Maestro é só para PC (o Maestro roda pelo Pinokio no seu computador).


In [ ]:
#@title 1. Instalar
USAR_DRIVE = False  #@param {type:"boolean"}
BRANCH = "claude/dazzling-hopper-f2l8o9"  #@param {type:"string"}
import base64, os, subprocess
REPO = "https://github.com/FelipeGomes070898/ARCHIVE-STUDIO-AI.git"
DEST = "/content/ARCHIVE-STUDIO-AI"

# Token do GitHub (Secrets do Colab) — necessário porque o repositório é privado. Nunca é impresso nem salvo no disco.
token = os.environ.get("GITHUB_TOKEN", "")
if not token:
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN") or ""
        print("Token do GitHub lido dos Secrets.")
    except ImportError:
        pass
    except Exception as e:
        tipo = type(e).__name__
        if "NotFound" in tipo:
            print("Segredo GITHUB_TOKEN não existe nos Secrets (ícone 🔑).")
        elif "Access" in tipo:
            print("GITHUB_TOKEN existe, mas o 'Acesso ao notebook' está desligado nos Secrets (ícone 🔑).")
        else:
            print(f"Não consegui ler os Secrets ({tipo}).")
if not token:
    from getpass import getpass
    token = getpass("Cole aqui seu token do GitHub (fica oculto) e aperte Enter: ").strip()
os.environ["GITHUB_TOKEN"] = token     # só nesta sessão do Colab, para as próximas execuções
auth = []
if token:
    basic = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    auth = ["-c", f"http.extraHeader=Authorization: Basic {basic}"]

def git(*args):
    r = subprocess.run(["git", *auth, *args], capture_output=True, text=True)
    if r.returncode != 0:
        msg = (r.stderr or r.stdout).replace(token, "***") if token else (r.stderr or r.stdout)
        hint = ("\n➡ Repositório privado: crie o segredo GITHUB_TOKEN (veja a primeira célula de texto) "
                "e ative o 'Acesso ao notebook'." if not token else
                "\n➡ Confira se o token tem acesso ao repositório (Contents: Read-only) e se o BRANCH existe.")
        raise SystemExit(f"Falha no git: {msg.strip()}{hint}")
    return r

if os.path.isdir(DEST + "/.git"):
    git("-C", DEST, "fetch", "-q", "origin", BRANCH)
    git("-C", DEST, "checkout", "-q", "-B", BRANCH, "FETCH_HEAD")
else:
    git("clone", "-q", "--depth", "1", "--branch", BRANCH, REPO, DEST)
print("Código baixado:", subprocess.run(["git", "-C", DEST, "log", "--oneline", "-1"], capture_output=True, text=True).stdout.strip())

%pip install -q -r /content/ARCHIVE-STUDIO-AI/requirements-gpu.txt
if USAR_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ["ARCHIVE_STUDIO_HOME"] = "/content/drive/MyDrive/archive_studio"
else:
    os.environ["ARCHIVE_STUDIO_HOME"] = "/content/archive_studio"
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "Sem GPU: ative em Ambiente de execução > Alterar o tipo"
print("Instalado. Projetos em", os.environ["ARCHIVE_STUDIO_HOME"])


In [ ]:
#@title 2. Chaves (dos Secrets do Colab, se existirem)
import os
try:
    from google.colab import userdata
    for k in ("GEMINI_API_KEY", "HF_TOKEN"):
        try:
            v = userdata.get(k)
            if v:
                os.environ[k] = v
                print(k, "carregada")
        except Exception:
            print(k, "não encontrada (opcional)")
except ImportError:
    pass

In [ ]:
#@title 3. Abrir o painel
import os, re, secrets, subprocess, sys, threading, time, urllib.request
os.chdir("/content/ARCHIVE-STUDIO-AI")
sys.path.insert(0, "/content/ARCHIVE-STUDIO-AI")
PORT, TOKEN = 8765, secrets.token_urlsafe(12)
from studio import server
threading.Thread(target=server.main, args=(["--port", str(PORT), "--no-browser", "--token", TOKEN],), daemon=True).start()
time.sleep(3)

# Link externo (túnel Cloudflare gratuito, protegido pelo token)
if not os.path.exists("/content/cloudflared"):
    urllib.request.urlretrieve("https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", "/content/cloudflared")
    os.chmod("/content/cloudflared", 0o755)
tun = subprocess.Popen(["/content/cloudflared", "tunnel", "--no-autoupdate", "--url", f"http://127.0.0.1:{PORT}"],
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = None
for _ in range(60):
    line = tun.stdout.readline()
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line or "")
    if m:
        url = m.group(0); break
threading.Thread(target=lambda: [None for _ in tun.stdout], daemon=True).start()  # esvazia o log do túnel
print("Link externo (abra em outra aba):", f"{url}/?t={TOKEN}" if url else "não disponível agora")

# Painel dentro do Colab
from google.colab import output
output.serve_kernel_port_as_iframe(PORT, path=f"/?t={TOKEN}", height=900)
print("Mantenha esta aba aberta enquanto produz.")